In [ ]:
# Install required packages:
# - gymnasium: the RL environment framework
# - ale-py: Atari Learning Environment bindings
# - autorom: downloads Atari ROMs needed to run the games
!pip install gymnasium ale-py autorom


In [ ]:
# Accept the Atari ROM license and install ROMs automatically
!AutoROM --accept-license


In [ ]:
# pygame: used internally by some gym environments for rendering
!pip install pygame


In [ ]:
# opencv: used to display frames in the play loop
!pip install opencv-python


In [ ]:
# Install PyTorch (CPU or GPU depending on your system)
%pip install torch torchvision torchaudio


In [ ]:
import gymnasium as gym
import ale_py
import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
# Register all ALE (Atari) environments with gymnasium so they can be created by name
gym.register_envs(ale_py)

# Training environment: uses RAM observations (128 bytes) instead of pixels.
# RAM is much cheaper to process and contains all the game state we need.
env = gym.make("ALE/Freeway-v5", obs_type="ram")


In [ ]:
# ---------------------------
# NEURAL NETWORK (MLP)
# ---------------------------
# A Multi-Layer Perceptron (fully connected network) that acts as our policy.
# It takes the 128 RAM bytes as input and outputs a score (logit) for each action.
# The action with the highest probability (after softmax) is what the agent does.

class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 128),  # Input layer: 128 RAM bytes → 128 hidden units
            nn.ReLU(),                   # Non-linear activation
            nn.Linear(128, 128),         # Hidden layer
            nn.ReLU(),
            nn.Linear(128, output_size)  # Output layer: one logit per action
        )

    def forward(self, x):
        return self.net(x)


# ---------------------------
# MODEL INITIALIZATION
# ---------------------------
input_size = 128                         # Atari RAM is always 128 bytes
output_size = env.action_space.n         # Freeway has 3 actions: noop, up, down

model = MLP(input_size, output_size)

# Adam optimizer: updates model weights during training
# lr=0.001 is the learning rate — controls how big each weight update step is
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# ---------------------------
# TRAINING CONFIG
# ---------------------------
# Set RESET_TRAINING = True to start fresh (ignore any saved checkpoint)
# Set RESET_TRAINING = False to resume training from where you left off
RESET_TRAINING = True

episode_rewards = []  # Tracks total reward per episode for monitoring progress
episodes = 3          # Number of episodes to train for (increase for real training)
start_episode = 0

# ---------------------------
# CHECKPOINT LOADING
# ---------------------------
if not RESET_TRAINING:
    try:
        checkpoint = torch.load("checkpoint.pth", weights_only=True)  # FIX: weights_only=True avoids security warning
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        start_episode = checkpoint["episode"] + 1
        print(f"Resuming from episode {start_episode}")
    except FileNotFoundError:  # FIX: only catch missing file, not all errors
        print("No checkpoint found, starting fresh.")
else:
    print("RESET_TRAINING = True → Starting from scratch (ignoring checkpoint)")

# ---------------------------
# TRAINING LOOP (REINFORCE / Policy Gradient)
# ---------------------------
# REINFORCE works by:
#   1. Running a full episode and recording actions + rewards
#   2. After the episode, updating the policy to make good actions more likely
#      and bad actions less likely, weighted by the reward they produced

for episode in range(start_episode, episodes):
    obs, _ = env.reset()

    log_probs = []  # Stores log probability of each action taken this episode
    rewards = []    # Stores the reward received at each step this episode
    prev_y = None   # Tracks the chicken's Y position from the previous step

    done = False

    # --- EPISODE ROLLOUT ---
    while not done:
        # Normalize RAM bytes from [0, 255] to [0.0, 1.0] for stable training
        ram_input = torch.tensor(obs, dtype=torch.float32).unsqueeze(0) / 255.0

        # Forward pass: get action logits from the model
        logits = model(ram_input)

        # Convert logits to probabilities using softmax
        probs = torch.softmax(logits, dim=1)

        # Sample an action from the probability distribution (allows exploration)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()

        # Record the log probability of the chosen action (used in REINFORCE loss)
        log_prob = dist.log_prob(action)

        # Take the action in the environment
        obs, _, term, trunc, _ = env.step(action.item())
        done = term or trunc

        # ---------------------------
        # CUSTOM REWARD FUNCTION
        # ---------------------------
        # The environment's built-in reward is only +1 when the chicken fully crosses.
        # That's too sparse to learn from, so we define a reward based on RAM byte 14,
        # which encodes the chicken's vertical position on screen.
        #   - Moving up (toward finish) → positive reward proportional to progress
        #   - Standing still or drifting slightly → neutral
        #   - Large drop (hit by car, reset to bottom) → strong penalty
        #   - Every step gets a small time penalty to encourage moving quickly

        chicken_y = int(obs[14])  # RAM byte 14 = chicken's vertical position

        if prev_y is None:
            # First step of the episode — no previous position to compare against
            reward = 0
        else:
            delta = chicken_y - prev_y  # Positive = moved up, negative = moved down

            if delta > 0:
                # Moved up toward the goal — reward proportional to distance gained
                reward = delta
            elif -10 <= delta <= 0:
                # Small or no movement — neutral (not penalized)
                reward = 0
            else:
                # Large downward jump — likely hit by a car and reset
                reward = delta - 5  # Extra -5 penalty on top of the position loss

            # Small time penalty applied every step to discourage loitering
            reward -= 0.1

        prev_y = chicken_y

        # FIX: Actually store log_prob and reward each step.
        # Without these appends, the lists stay empty and the model never trains.
        log_probs.append(log_prob)
        rewards.append(reward)

    # ---------------------------
    # POLICY UPDATE (REINFORCE Loss)
    # ---------------------------
    # Loss = -sum(log_prob * reward) for each step
    # Steps with positive reward → loss decreases → those actions become more likely
    # Steps with negative reward → loss increases → those actions become less likely
    loss = 0
    for lp, r in zip(log_probs, rewards):  # FIX: renamed loop vars to avoid shadowing outer log_prob/reward
        loss += -lp * r

    optimizer.zero_grad()   # Clear gradients from the previous step
    loss.backward()         # Compute gradients via backpropagation
    optimizer.step()        # Update model weights

    total_reward = sum(rewards)
    episode_rewards.append(total_reward)
    print(f"Episode {episode} | Total reward: {total_reward:.2f}")

    # Save a checkpoint every 10 episodes so training can be resumed later
    if episode % 10 == 0 and episode > 0:
        torch.save({
            "episode": episode,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
        }, "checkpoint.pth")
        print(f"  → Checkpoint saved at episode {episode}")

# ---------------------------
# SAVE FINAL MODEL
# ---------------------------
torch.save(model.state_dict(), "freeway_mlp.pth")
print("Training complete. Model saved to freeway_mlp.pth")

env.close()


In [ ]:
# ---------------------------
# PLAY LOOP (Inference / Visualization)
# ---------------------------
# Two environments are used in sync:
#   env_vis → returns RGB frames for display (no RAM obs)
#   env_ram → returns RAM observations for the model to make decisions
# Both receive the same action each step to stay in sync.

env_vis = gym.make("ALE/Freeway-v5", render_mode="rgb_array")
env_ram = gym.make("ALE/Freeway-v5", obs_type="ram")

# Use the same seed for both environments so they start in the same state
obs_vis, _ = env_vis.reset(seed=42)
obs_ram, _ = env_ram.reset(seed=42)  # FIX: matching seeds reduce env desync risk

# Load the trained model weights
# weights_only=True avoids a PyTorch security warning about arbitrary pickle execution
model.load_state_dict(torch.load("freeway_mlp.pth", weights_only=True))  # FIX: weights_only=True
model.eval()  # Switch to inference mode (disables dropout etc. if present)

while True:
    # Prepare RAM input: normalize to [0, 1] and add batch dimension
    ram_input = np.array(obs_ram, dtype=np.float32) / 255.0
    ram_input = torch.tensor(ram_input).unsqueeze(0)

    # Greedy action selection: pick the action with the highest score (no sampling)
    # torch.no_grad() skips gradient computation since we're not training here
    with torch.no_grad():
        logits = model(ram_input)
        action = torch.argmax(logits, dim=1).item()

    # Step both environments with the same action to keep them in sync
    obs_vis, _, term1, trunc1, _ = env_vis.step(action)
    obs_ram, _, term2, trunc2, _ = env_ram.step(action)

    # Display the visual frame using OpenCV
    frame_bgr = cv2.cvtColor(obs_vis.copy(), cv2.COLOR_RGB2BGR)  # OpenCV uses BGR
    cv2.imshow("Freeway AI", frame_bgr)
    time.sleep(0.03)  # ~33fps cap

    # Press 'q' to quit the viewer
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    # FIX: reset if EITHER environment ends, not just the visual one
    # This prevents the environments from drifting out of sync
    if term1 or trunc1 or term2 or trunc2:
        obs_vis, _ = env_vis.reset(seed=42)
        obs_ram, _ = env_ram.reset(seed=42)

env_vis.close()
env_ram.close()
cv2.destroyAllWindows()  # FIX: properly close the OpenCV window on exit
